<a href="https://colab.research.google.com/github/Prateek-Kumar-Anand/Experimenting-with-AI/blob/main/emotion_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**DATA PREPEARATION**

*IMPORTING LIBRARIES*

In [1]:
import pandas
import torch
from sklearn.model_selection import train_test_split

*LOADING DATA*

In [ ]:
data = "https://raw.githubusercontent.com/Prateek-Kumar-Anand/Experimenting-with-AI/refs/heads/main/custom_datasets/emotion_dataset_100k.csv"
df = pandas.read_csv(data)
print(df.head())
print(df.shape)

   id                                               text    label  label_id
0   1         Ok, I updated my laptop. regular afternoon  neutral         6
1   2                               Rn, on my way lol. 👌  neutral         6
2   3  My teacher marked me wrong when I was right an...    anger         2
3   4  WOOO THE WHOLE FAMILY IS AT MY PLACE TONIGHT. ...      joy         0
4   5                                      Today, noted.  neutral         6
(99976, 4)


*DATA PREPROCESSING*

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"])

validate_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"])

# Directly 3 CSV files
train_df.to_csv("train.csv", index=False)
validate_df.to_csv("validate.csv", index=False)
test_df.to_csv("test.csv", index=False)

In [ ]:
train = pandas.read_csv("train.csv")
validate = pandas.read_csv("validate.csv")
test = pandas.read_csv("test.csv")

In [ ]:
print(train_df.shape)
print(validate_df.shape)
print(test_df.shape)

(79980, 4)
(9998, 4)
(9998, 4)


In [ ]:
train.describe()
validate.describe()
test.describe()

,id,label_id
count,9998.000000,9998.0000
mean,49860.263353,2.9992
std,28952.584073,2.0000
min,15.000000,0.0000
25%,24765.000000,1.0000
50%,49579.000000,3.0000
75%,75083.000000,5.0000
max,99966.000000,6.0000


*DATA PROCESSING*

In [ ]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(train_df)
validate_dataset = Dataset.from_pandas(validate_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
print(train_dataset)
print(validate_dataset)
print(test_dataset)

Dataset({
    features: ['id', 'text', 'label', 'label_id', '__index_level_0__'],
    num_rows: 79980
})
Dataset({
    features: ['id', 'text', 'label', 'label_id', '__index_level_0__'],
    num_rows: 9998
})
Dataset({
    features: ['id', 'text', 'label', 'label_id', '__index_level_0__'],
    num_rows: 9998
})


In [ ]:
dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": validate_dataset,
    "test": test_dataset
})

In [ ]:
print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'label_id', '__index_level_0__'],
        num_rows: 79980
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'label_id', '__index_level_0__'],
        num_rows: 9998
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'label_id', '__index_level_0__'],
        num_rows: 9998
    })
})


*IMPORTING TOKENIZERE*

In [ ]:
!pip install transformers

In [ ]:
from transformers import BertTokenizer, BertModel
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset_dict = dataset_dict.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/79980 [00:00<?, ? examples/s]

Map:   0%|          | 0/9998 [00:00<?, ? examples/s]

Map:   0%|          | 0/9998 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset_dict.set_format("torch")

In [ ]:
print(tokenized_dataset_dict["train"][0])

{'id': tensor(13642), 'text': "Wtf! I'm shook. we won and we were the underdogs. brooo what the hell", 'label': 'surprise', 'label_id': tensor(4), '__index_level_0__': tensor(13641), 'input_ids': tensor([  101,  1059, 24475,   999,  1045,  1005,  1049,  3184,  1012,  2057,
         2180,  1998,  2057,  2020,  1996,  2104, 16168,  2015,  1012, 22953,
         9541,  2054,  1996,  3109,   102,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,   

In [ ]:
tokenized_dataset_dict = tokenized_dataset_dict.remove_columns([
    "id",
    "text",
    "label",
    "__index_level_0__"

])

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    tokenized_dataset_dict["train"],
    batch_size=16,
    shuffle=True
)

validate_loader = DataLoader(
    tokenized_dataset_dict["validation"],
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    tokenized_dataset_dict["test"],
    batch_size=16,
    shuffle=False
)

In [ ]:
batch = next(iter(train_loader))

print(batch.keys())

dict_keys(['label_id', 'input_ids', 'token_type_ids', 'attention_mask'])


In [ ]:
tokenized_dataset_dict.set_format("torch")

In [ ]:
print(tokenized_dataset_dict["train"].column_names)

['label_id', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
batch = next(iter(train_loader))

print(batch.keys())
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["token_type_ids"].shape)
print(batch["label_id"].shape)

dict_keys(['label_id', 'input_ids', 'token_type_ids', 'attention_mask'])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16])


In [ ]:
print(tokenized_dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['label_id', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 79980
    })
    validation: Dataset({
        features: ['label_id', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9998
    })
    test: Dataset({
        features: ['label_id', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9998
    })
})


In [ ]:
print(batch)

{'label_id': tensor([3, 1, 0, 4, 6, 0, 5, 4, 2, 2, 5, 5, 5, 3, 1, 3]), 'input_ids': tensor([[  101,  1059, 24475,  ...,     0,     0,     0],
        [  101,  9826,  1045,  ...,     0,     0,     0],
        [  101,  8038,  2100,  ...,     0,     0,     0],
        ...,
        [  101,  6616,  1010,  ...,     0,     0,     0],
        [  101,  1045,  2246,  ...,     0,     0,     0],
        [  101,  2026,  3042,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}


In [ ]:
import torch
import torch.nn as nn

class EmotionModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64)
        self.fc1 = nn.Linear(64, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 7)

    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids)
        mask = attention_mask.unsqueeze(-1).float()
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

model = EmotionModel(tokenizer.vocab_size)

In [ ]:
import torch
import torch.nn as nn

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

for epoch in range(7):

    # TRAINING
    model.train()
    train_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["label_id"].long()

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # VALIDATION
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in validate_loader:
            outputs = model(
                batch["input_ids"],
                batch["attention_mask"]
            )

            labels = batch["label_id"].long()
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    print(f"Epoch: {epoch + 1}")
    print(f"Train Loss: {train_loss / len(train_loader):.4f}")
    print(f"Validation Loss: {val_loss / len(validate_loader):.4f}")
    print(f"Validation Accuracy: {correct / total:.4f}")

Epoch: 1
Train Loss: 1.2101
Validation Loss: 0.6103
Validation Accuracy: 0.8262
Epoch: 2
Train Loss: 0.3738
Validation Loss: 0.2184
Validation Accuracy: 0.9392
Epoch: 3
Train Loss: 0.1518
Validation Loss: 0.1040
Validation Accuracy: 0.9705
Epoch: 4
Train Loss: 0.0768
Validation Loss: 0.0589
Validation Accuracy: 0.9834
Epoch: 5
Train Loss: 0.0451
Validation Loss: 0.0368
Validation Accuracy: 0.9899
Epoch: 6
Train Loss: 0.0293
Validation Loss: 0.0248
Validation Accuracy: 0.9931
Epoch: 7
Train Loss: 0.0206
Validation Loss: 0.0186
Validation Accuracy: 0.9951


In [ ]:
torch.save(model.state_dict(), "emotion_model.pth")

In [ ]:
model2 = EmotionModel(tokenizer.vocab_size)

model2.load_state_dict(
    torch.load("emotion_model.pth", map_location="cpu")
)

model2.eval()

EmotionModel(
  (embedding): Embedding(30522, 64)
  (fc1): Linear(in_features=64, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=7, bias=True)
)

In [ ]:
text = input()
inputs = tokenizer(
    text,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=128
)
with torch.no_grad():
    logits = model2(
        inputs["input_ids"],
        inputs["attention_mask"]
    )
predicted_class = torch.argmax(logits, dim=1).item()
print("Predicted class:", predicted_class)

I am horny
Predicted class: 0


In [ ]:
id_to_label = dict(
    zip(df["label_id"], df["label"])
)

print("Predicted emotion:", id_to_label[predicted_class])

Predicted emotion: joy
